# Voting — let diverse classifiers vote

> Tutorial pair for [`voting.py`](voting.py).

## 1. Intuition
The simplest ensemble: train a few good, *different* classifiers and let them
vote. **Hard voting** counts predicted labels (majority wins). **Soft voting**
averages the predicted *probabilities* and takes the argmax — so a model that is
very confident and right can override several models that are unsure and wrong.
No meta-learner, no resampling: just aggregate. It works because diverse models
make *independent* mistakes that tend to cancel.

## 2. Concept (the slide)
- **Hard voting:** $\hat y=\text{mode}\{h_b(x)\}$ — plurality of predicted labels.
- **Soft voting:** $\hat y=\arg\max_k \frac1B\sum_b p_b(k\mid x)$ — average
  probabilities (uses confidence, usually better).
- **Weighted voting:** trust some models more via weights $w_b$.
- **Requirement:** the base models should be **diverse** (different inductive
  biases) and individually *better than random* — else voting can hurt.
- Contrast: voting fixes the combiner (mean/mode); **stacking learns** it.

## 3. Math derivation

**Why voting helps — the independent-errors argument.** Suppose $B$ classifiers
each have accuracy $p>\tfrac12$ on a binary problem and make **independent**
errors. Majority vote is correct when more than half are correct; the number
correct is $\text{Binomial}(B,p)$, so

$$\Pr[\text{majority correct}]=\sum_{k=\lceil B/2\rceil}^{B}\binom{B}{k}p^k(1-p)^{B-k}
  \xrightarrow{B\to\infty} 1 .$$

(Condorcet's jury theorem.) E.g. $B=15$ voters at $p=0.7$ give $\approx0.95$
ensemble accuracy. The catch is the word **independent**: if the classifiers are
identical their votes are perfectly correlated and the ensemble equals one model.
So **diversity is the whole game**.

**Soft vs hard.** Hard voting discards confidence. Soft voting averages the class
posteriors,

$$\bar p(k\mid x)=\frac{1}{\sum_b w_b}\sum_{b=1}^{B} w_b\, p_b(k\mid x),\qquad
  \hat y=\arg\max_k \bar p(k\mid x),$$

which is the Bayes-optimal combination if the $p_b$ are calibrated estimates of
the same posterior. Concretely, if two models say class A with probability $0.55$
and one says class B with probability $0.95$, hard voting picks A (2 vs 1) but
soft voting picks B ($\bar p_B=0.95/3\approx0.32$ vs $\bar p_A=1.10/3\approx0.37$
— A still wins here, but tilt the confidences and soft voting flips, correctly
following the confident model). In practice soft voting $\ge$ hard voting when
the base probabilities are reasonably calibrated.

**Bias–variance view.** Like bagging, averaging *uncorrelated* predictors reduces
variance ($\operatorname{Var}(\bar p)\to\rho\sigma^2$ as $B\to\infty$); unlike
bagging, the diversity here comes from using **different model families** rather
than resampling one family. Weighted voting lets you down-weight a weak/redundant
member.

## 4. NumPy implementation (hard / soft / weighted voting over diverse bases)

In [ ]:
# ===== actual implementation from voting.py =====
from __future__ import annotations

import numpy as np

SEED = 0

class _LogReg:
    """Softmax (multinomial) logistic regression via gradient descent."""

    def __init__(self, lr=0.3, n_iter=400, l2=1e-3):
        self.lr, self.n_iter, self.l2 = lr, n_iter, l2

    @staticmethod
    def _softmax(Z):
        Z = Z - Z.max(1, keepdims=True); E = np.exp(Z); return E / E.sum(1, keepdims=True)

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y).astype(int)
        n, d = X.shape
        self.classes_ = np.unique(y); K = len(self.classes_)
        Y = np.eye(K)[np.searchsorted(self.classes_, y)]
        self.W = np.zeros((d, K)); self.b = np.zeros(K)
        for _ in range(self.n_iter):
            P = self._softmax(X @ self.W + self.b)
            self.W -= self.lr * (X.T @ (P - Y) / n + self.l2 * self.W)
            self.b -= self.lr * (P - Y).mean(0)
        return self

    def predict_proba(self, X):
        return self._softmax(np.asarray(X, float) @ self.W + self.b)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]

class _Tree:
    """Shallow CART classifier; leaves hold class proportions (probabilities)."""

    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth, self.min_samples_split = max_depth, min_samples_split

    def _best_split(self, X, y):
        n, d = X.shape; best = (np.inf, None, None)
        for f in range(d):
            order = np.argsort(X[:, f], kind="mergesort")
            xs = X[order, f]; valid = xs[:-1] != xs[1:]
            if not valid.any():
                continue
            cl_n = np.arange(1, n); cr_n = n - cl_n
            oh = np.eye(self._K)[y[order].astype(int)]
            cum = np.cumsum(oh, axis=0); tot = cum[-1]
            cl, cr = cum[:-1], tot - cum[:-1]
            gl = 1 - ((cl / cl_n[:, None]) ** 2).sum(1)
            gr = 1 - ((cr / cr_n[:, None]) ** 2).sum(1)
            s = np.where(valid, (cl_n * gl + cr_n * gr) / n, np.inf)
            j = int(np.argmin(s))
            if s[j] < best[0]:
                best = (s[j], f, (xs[j] + xs[j + 1]) / 2)
        return best

    def _build(self, X, y, depth):
        if (len(y) < self.min_samples_split or depth >= self.max_depth or
                len(np.unique(y)) == 1):
            return ("leaf", np.bincount(y.astype(int), minlength=self._K) / len(y))
        _, f, t = self._best_split(X, y)
        if f is None:
            return ("leaf", np.bincount(y.astype(int), minlength=self._K) / len(y))
        m = X[:, f] <= t
        return ("node", f, t, self._build(X[m], y[m], depth + 1),
                self._build(X[~m], y[~m], depth + 1))

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        self.classes_ = np.unique(y); self._K = int(y.max()) + 1
        self.root = self._build(X, y, 0)
        return self

    def _one(self, x, node):
        if node[0] == "leaf":
            return node[1]
        _, f, t, l, r = node
        return self._one(x, l if x[f] <= t else r)

    def predict_proba(self, X):
        return np.array([self._one(x, self.root) for x in np.asarray(X, float)])

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]

class _GaussianNB:
    """Gaussian Naive Bayes — independent per-feature Gaussians per class."""

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y).astype(int)
        self.classes_ = np.unique(y)
        self.theta, self.var, self.prior = [], [], []
        for c in self.classes_:
            Xc = X[y == c]
            self.theta.append(Xc.mean(0))
            self.var.append(Xc.var(0) + 1e-9)        # variance floor
            self.prior.append(len(Xc) / len(X))
        self.theta = np.array(self.theta); self.var = np.array(self.var)
        self.prior = np.array(self.prior)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, float)
        # log P(c|x) ∝ log prior - 1/2 sum[log(2πσ²) + (x-μ)²/σ²]
        logp = []
        for k in range(len(self.classes_)):
            ll = -0.5 * (np.log(2 * np.pi * self.var[k])
                         + (X - self.theta[k]) ** 2 / self.var[k]).sum(1)
            logp.append(np.log(self.prior[k]) + ll)
        logp = np.array(logp).T
        logp -= logp.max(1, keepdims=True)
        P = np.exp(logp)
        return P / P.sum(1, keepdims=True)

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(1)]

def demo():
    np.random.seed(SEED)
    from sklearn.datasets import make_classification

    X, y = make_classification(n_samples=600, n_features=12, n_informative=7,
                               n_redundant=2, n_classes=3, n_clusters_per_class=1,
                               random_state=SEED)
    Xtr, ytr, Xte, yte = X[:450], y[:450], X[450:], y[450:]

    def fresh():
        return [("lr", _LogReg()), ("dt", _Tree(max_depth=5)), ("gnb", _GaussianNB())]

    # individual base accuracies
    for name, m in fresh():
        m.fit(Xtr, ytr)
        print(f"[base] {name:4s} acc={np.mean(m.predict(Xte) == yte):.3f}")

    hard = VotingNumPy(fresh(), voting="hard").fit(Xtr, ytr)
    soft = VotingNumPy(fresh(), voting="soft").fit(Xtr, ytr)
    print(f"[vote] hard acc={np.mean(hard.predict(Xte) == yte):.3f}")
    print(f"[vote] soft acc={np.mean(soft.predict(Xte) == yte):.3f}")

    # weighted soft voting: trust logistic regression more
    wsoft = VotingNumPy(fresh(), voting="soft", weights=[2, 1, 1]).fit(Xtr, ytr)
    print(f"[vote] soft weighted[2,1,1] acc={np.mean(wsoft.predict(Xte) == yte):.3f}")

    skh = sklearn_reference(Xtr, ytr, voting="hard")
    sks = sklearn_reference(Xtr, ytr, voting="soft")
    print(f"[sk  ] hard acc={np.mean(skh.predict(Xte) == yte):.3f}  "
          f"soft acc={np.mean(sks.predict(Xte) == yte):.3f}")


class VotingNumPy:
    """Hard or soft voting over a list of fitted-on-fit base classifiers.

    estimators : list of (name, fresh_unfitted_model) tuples.
    voting     : "hard" (majority label) or "soft" (mean probabilities).
    weights    : optional per-estimator weights (soft voting).
    """

    def __init__(self, estimators, voting="soft", weights=None):
        self.estimators = estimators
        self.voting = voting
        self.weights = weights

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y)
        self.classes_ = np.unique(y)
        self.models_ = [(name, m.fit(X, y)) for name, m in self.estimators]
        return self

    def _weights(self):
        if self.weights is None:
            return np.ones(len(self.models_))
        return np.asarray(self.weights, float)

    def predict_proba(self, X):
        w = self._weights()
        P = np.zeros((len(np.asarray(X, float)), len(self.classes_)))
        for (_, m), wi in zip(self.models_, w):
            P += wi * m.predict_proba(X)         # weighted average of probabilities
        return P / w.sum()

    def predict(self, X):
        if self.voting == "soft":
            return self.classes_[self.predict_proba(X).argmax(1)]
        # hard voting: weighted plurality of predicted labels
        w = self._weights()
        n = len(np.asarray(X, float))
        tally = np.zeros((n, len(self.classes_)))
        idx = {c: i for i, c in enumerate(self.classes_)}
        for (_, m), wi in zip(self.models_, w):
            pred = m.predict(X)
            for i in range(n):
                tally[i, idx[pred[i]]] += wi
        return self.classes_[tally.argmax(1)]

## 5. Reference / cross-check — why not PyTorch?

Voting is a thin aggregation layer over already-trained, heterogeneous
classifiers (logistic regression, a tree, Gaussian NB). The combiner is a fixed
`argmax` of averaged probabilities or a plurality of labels — nothing to
differentiate — so an idiomatic PyTorch model is not the natural tool. We
cross-check against scikit-learn's `VotingClassifier`.

In [ ]:
# ===== actual implementation from voting.py =====
def sklearn_reference(X, y, voting="soft"):
    from sklearn.ensemble import VotingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.naive_bayes import GaussianNB
    from sklearn.tree import DecisionTreeClassifier
    estimators = [("lr", LogisticRegression(max_iter=500)),
                  ("dt", DecisionTreeClassifier(max_depth=5, random_state=SEED)),
                  ("gnb", GaussianNB())]
    return VotingClassifier(estimators, voting=voting).fit(X, y)

## 6. Train — base accuracies vs hard / soft / weighted voting

In [ ]:
demo()

## 7. Visualization — base vs voted decision regions

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import voting as M

X, y = make_moons(n_samples=300, noise=0.3, random_state=0)
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-.5, X[:,0].max()+.5, 200),
                     np.linspace(X[:,1].min()-.5, X[:,1].max()+.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

def fresh():
    return [("lr", M._LogReg()), ("dt", M._Tree(max_depth=5)), ("gnb", M._GaussianNB())]

models = {
    "logreg": M._LogReg().fit(X, y),
    "tree": M._Tree(max_depth=5).fit(X, y),
    "soft vote": M.VotingNumPy(fresh(), voting="soft").fit(X, y),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    zz = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap="coolwarm")
    ax.scatter(X[:,0], X[:,1], c=y, s=12, edgecolor="k", cmap="coolwarm")
    ax.set_title(name)   # voted boundary blends linear + axis-aligned biases
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Soft voting usually beats hard voting** — but only if the base probabilities
  are roughly *calibrated*; uncalibrated models can mislead the average.
- **Diversity is essential** (Condorcet): correlated/identical models give no
  gain. Mix model families and feature views.
- A base model worse than random *drags the ensemble down* — drop it or
  down-weight it.
- Voting has no learned combiner; when you want the blend itself optimized, use
  **stacking**. When you want variance reduction from *one* family, use bagging.